In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt
import sys
from pathlib import Path

In [ ]:
# dss and ghost corrected earth image 
obs_path = "/home/bekah/m3-pipeline-dev/data/moon_dss_ghost_corr.fits"

with fits.open(obs_path) as hdul:
        obs_image = hdul[0].data
    

In [ ]:
obs_image.shape

In [ ]:


line_range = range(550, 700)  

records = []

for line in line_range:
    obs = obs_image[line, :, :]

    obs = obs - obs_image[1054, :, :]
    
    n_bands = obs.shape[0]  

    denom = np.percentile(obs[ :, 118:135], 90, axis=1) 
    num = np.percentile(obs[ :, 149:170], 90, axis=1)  
    ratio = num  / denom

    for band in range(86):
        records.append({
                    "line": line,
                    "band": band,
                    "ratio": ratio[band],
                })

df = pd.DataFrame(records)
df.to_csv("median_ratios_per_band_global_earthdss_2.csv", index=False)

In [ ]:
# plot each obs line 
band_defs = {
    "left": slice(4, 7),
}

fig, ax = plt.subplots(figsize=(10, 6))
for line_id, group in df.groupby("line"):
    group = group.sort_values("band")  # important: plot() connects points in order
    ax.plot(group["band"], group["ratio"], alpha=0.3, linewidth=1)

ax.set_xlabel("band")
ax.set_ylabel("median ratio")
plt.title("band ratios for multiple global obs")
plt.ylim(-.1, .3)
plt.tight_layout()
plt.savefig("global_med_sl_ratio_all_2.png")

In [ ]:
sub

In [ ]:
# median per band across all obs 

summary = df.groupby(["band"])["ratio"].median().reset_index()

plt.plot(summary["band"], summary["ratio"], linewidth=2, linestyle=":")
plt.title("band ratios for global, median of earth dss") 
plt.xlabel("x")
plt.ylabel("median ratio")
plt.axhline(0)
plt.ylim(-.1,.3)

plt.savefig("global_med_sl_ratio_earthdss_2.png")

In [ ]:
l0_l1b_parent = Path("/home/bekah/m3-pipeline-dev")
if str(l0_l1b_parent) not in sys.path:
    sys.path.insert(0, str(l0_l1b_parent))
    
from l0_l1b_l2.l1b_utils.scattered_light import apply_scattered_light_corr

In [ ]:
obs_image = apply_scattered_light_corr(obs_image, 'G', True, sigma=11.0) 

In [ ]:
fits.writeto(
            f"{l0_l1b_parent}/data/earth_sl_corr.fits",
            obs_image, 
            overwrite=True
        )